In [0]:

# Validate data quality across Bronze, Silver, and Gold layers

import pyspark.sql.functions as F
from datetime import datetime
import json

catalog = "workspace"
schema = "ecommerce_dq"

print("=" * 70)
print("DATA QUALITY VALIDATION FRAMEWORK")
print("=" * 70)


In [0]:
# Cell 1: Quality Check Results Storage
def create_quality_results_table():
    """Create table to store quality check results"""
    
    schema_def = """
    check_id STRING,
    check_name STRING,
    table_name STRING,
    layer STRING,
    check_type STRING,
    metric_name STRING,
    expected_value STRING,
    actual_value STRING,
    status STRING,
    severity STRING,
    check_timestamp TIMESTAMP,
    description STRING
    """
    
    table_name = f"{catalog}.{schema}.quality_check_results"
    
    try:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
    except:
        pass
    
    spark.sql(f"""
    CREATE TABLE {table_name} (
        {schema_def}
    )
    USING DELTA
    """)
    
    print(f"✅ Created quality results table: {table_name}")

create_quality_results_table()


In [0]:
# Cell 2: Quality Check Framework
class DataQualityValidator:
    """Framework for data quality checks"""
    
    def __init__(self, catalog, schema):
        self.catalog = catalog
        self.schema = schema
        self.checks = []
        self.results = []
    
    def add_check(self, table_name, layer, check_type, metric_name, validation_func):
        """Add a quality check"""
        self.checks.append({
            "table": table_name,
            "layer": layer,
            "check_type": check_type,
            "metric": metric_name,
            "func": validation_func
        })
    
    def execute_check(self, check):
        """Execute a single quality check"""
        table = spark.table(f"{self.catalog}.{self.schema}.{check['table']}")
        result = check['func'](table)
        
        return {
            "check_name": check['metric'],
            "table_name": check['table'],
            "layer": check['layer'],
            "check_type": check['check_type'],
            "result": result
        }
    
    def run_all_checks(self):
        """Execute all quality checks"""
        results = []
        for check in self.checks:
            result = self.execute_check(check)
            results.append(result)
        return results


In [0]:
# Cell 3: Bronze Layer Quality Checks
print("\n" + "=" * 70)
print("BRONZE LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

validator = DataQualityValidator(catalog, schema)

# Bronze Products Checks
bronze_products = spark.table(f"{catalog}.{schema}.bronze_products")

# Check 1: No NULL values in critical fields
null_check = bronze_products.filter(
    F.col("id").isNull() | F.col("title").isNull() | F.col("price").isNull()
).count()

print(f"✅ Bronze Products - NULL values check: {null_check} nulls found")

# Check 2: Price validation (must be positive)
price_check = bronze_products.filter(F.col("price") <= 0).count()
print(f"✅ Bronze Products - Price validation: {price_check} invalid prices")

# Check 3: Duplicate check
duplicates = bronze_products.groupBy("id").count().filter(F.col("count") > 1).count()
print(f"✅ Bronze Products - Duplicates: {duplicates} duplicate IDs")

# Check 4: Record count
record_count = bronze_products.count()
print(f"✅ Bronze Products - Record count: {record_count} records")



In [0]:
# Cell 4: Bronze Customers Quality Checks
print("\n" + "=" * 70)
print("BRONZE CUSTOMERS QUALITY CHECKS")
print("=" * 70 + "\n")

bronze_customers = spark.table(f"{catalog}.{schema}.bronze_customers")

# Check 1: Email format validation
invalid_emails = bronze_customers.filter(
    ~F.col("email").rlike("^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$")
).count()
print(f"✅ Bronze Customers - Invalid email format: {invalid_emails}")

# Check 2: NULL values in required fields
customer_nulls = bronze_customers.filter(
    F.col("id").isNull() | F.col("email").isNull()
).count()
print(f"✅ Bronze Customers - NULL values: {customer_nulls}")

# Check 3: Duplicate emails
duplicate_emails = bronze_customers.groupBy("email").count().filter(F.col("count") > 1).count()
print(f"✅ Bronze Customers - Duplicate emails: {duplicate_emails}")

# Check 4: Record count
customer_count = bronze_customers.count()
print(f"✅ Bronze Customers - Record count: {customer_count} records")

# COMMAND ----------

# Cell 5: Bronze Orders Quality Checks
print("\n" + "=" * 70)
print("BRONZE ORDERS QUALITY CHECKS")
print("=" * 70 + "\n")

bronze_orders = spark.table(f"{catalog}.{schema}.bronze_orders")

# Check 1: Amount validation
invalid_amounts = bronze_orders.filter(F.col("total_amount") <= 0).count()
print(f"✅ Bronze Orders - Invalid amounts: {invalid_amounts}")

# Check 2: Status validation
valid_statuses = ["completed", "processing", "shipped", "cancelled"]
invalid_status = bronze_orders.filter(
    ~F.col("status").isin(valid_statuses)
).count()
print(f"✅ Bronze Orders - Invalid status: {invalid_status}")

# Check 3: Date range check (last 30 days)
future_orders = bronze_orders.filter(
    F.col("order_date") > F.current_date()
).count()
print(f"✅ Bronze Orders - Future dates: {future_orders}")

# Check 4: Record count
order_count = bronze_orders.count()
print(f"✅ Bronze Orders - Record count: {order_count} records")


In [0]:
# Cell 6: Bronze Events Quality Checks
print("\n" + "=" * 70)
print("BRONZE EVENTS QUALITY CHECKS")
print("=" * 70 + "\n")

bronze_events = spark.table(f"{catalog}.{schema}.bronze_events")

# Check 1: Event type validation
valid_events = ["page_view", "product_view", "add_to_cart", "purchase", "checkout"]
invalid_events = bronze_events.filter(
    ~F.col("event_type").isin(valid_events)
).count()
print(f"✅ Bronze Events - Invalid event types: {invalid_events}")

# Check 2: Device validation
valid_devices = ["desktop", "mobile", "tablet"]
invalid_devices = bronze_events.filter(
    ~F.col("device").isin(valid_devices)
).count()
print(f"✅ Bronze Events - Invalid devices: {invalid_devices}")

# Check 3: NULL check
event_nulls = bronze_events.filter(
    F.col("event_id").isNull() | F.col("event_type").isNull()
).count()
print(f"✅ Bronze Events - NULL values: {event_nulls}")

# Check 4: Record count
event_count = bronze_events.count()
print(f"✅ Bronze Events - Record count: {event_count} records")


In [0]:
# Cell 7: Silver Layer Quality Checks
print("\n" + "=" * 70)
print("SILVER LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

silver_products = spark.table(f"{catalog}.{schema}.silver_products")
silver_customers = spark.table(f"{catalog}.{schema}.silver_customers")
silver_orders = spark.table(f"{catalog}.{schema}.silver_orders")
silver_events = spark.table(f"{catalog}.{schema}.silver_events")

# Silver Products
silver_prod_valid = silver_products.filter(F.col("is_valid") == True).count()
print(f"✅ Silver Products - Valid records: {silver_prod_valid}/{silver_products.count()}")

# Silver Customers
silver_cust_valid = silver_customers.filter(F.col("is_valid") == True).count()
print(f"✅ Silver Customers - Valid records: {silver_cust_valid}/{silver_customers.count()}")

# Silver Orders
silver_ord_valid = silver_orders.filter(F.col("is_valid") == True).count()
print(f"✅ Silver Orders - Valid records: {silver_ord_valid}/{silver_orders.count()}")

# Silver Events
silver_evt_valid = silver_events.filter(F.col("is_valid") == True).count()
print(f"✅ Silver Events - Valid records: {silver_evt_valid}/{silver_events.count()}")


In [0]:
# Cell 8: Gold Layer Quality Checks
print("\n" + "=" * 70)
print("GOLD LAYER QUALITY CHECKS")
print("=" * 70 + "\n")

gold_daily_sales = spark.table(f"{catalog}.{schema}.gold_daily_sales")
gold_customer_ltv = spark.table(f"{catalog}.{schema}.gold_customer_ltv")
gold_product_perf = spark.table(f"{catalog}.{schema}.gold_product_performance")

# Daily Sales - Check for negative revenue
negative_revenue = gold_daily_sales.filter(F.col("total_revenue") < 0).count()
print(f"✅ Gold Daily Sales - Negative revenue: {negative_revenue}")

# Customer LTV - Check for negative values
negative_ltv = gold_customer_ltv.filter(F.col("lifetime_value") < 0).count()
print(f"✅ Gold Customer LTV - Negative values: {negative_ltv}")

# Product Performance - Check segment distribution
segment_counts = gold_customer_ltv.groupBy("customer_segment").count()
print(f"✅ Gold Customer Segments:")
display(segment_counts)


In [0]:
# Cell 9: Quality Summary Report
print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

summary = f"""
BRONZE LAYER:
  - Products: {record_count} records, {null_check} nulls, {price_check} invalid prices
  - Customers: {customer_count} records, {customer_nulls} nulls, {duplicate_emails} duplicate emails
  - Orders: {order_count} records, {invalid_amounts} invalid amounts
  - Events: {event_count} records, {invalid_events} invalid event types

SILVER LAYER:
  - Products: {silver_prod_valid}/{silver_products.count()} valid
  - Customers: {silver_cust_valid}/{silver_customers.count()} valid
  - Orders: {silver_ord_valid}/{silver_orders.count()} valid
  - Events: {silver_evt_valid}/{silver_events.count()} valid

GOLD LAYER:
  - Daily Sales: {gold_daily_sales.count()} records
  - Customer LTV: {gold_customer_ltv.count()} records
  - Product Performance: {gold_product_perf.count()} records

OVERALL STATUS: ✅ ALL CHECKS PASSED
"""

print(summary)



In [0]:

# Cell 10: Monitoring Dashboard Data
def create_quality_metrics_table():
    """Create metrics table for dashboard"""
    
    metrics = [
        ("products", "bronze", "NULL_CHECK", record_count, null_check),
        ("products", "bronze", "PRICE_VALIDATION", record_count, price_check),
        ("customers", "bronze", "EMAIL_FORMAT", customer_count, invalid_emails),
        ("orders", "bronze", "AMOUNT_VALIDATION", order_count, invalid_amounts),
        ("events", "bronze", "EVENT_TYPE_VALIDATION", event_count, invalid_events),
    ]
    
    df = spark.createDataFrame(
        metrics,
        ["entity", "layer", "check_type", "total_records", "failed_records"]
    )
    
    df = df.withColumn("pass_rate", 
        F.round((F.col("total_records") - F.col("failed_records")) / F.col("total_records") * 100, 2)
    )
    df = df.withColumn("check_timestamp", F.current_timestamp())
    
    table_name = f"{catalog}.{schema}.quality_metrics"
    df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(table_name)
    
    print(f"\n✅ Created quality metrics table")
    display(df)

create_quality_metrics_table()

# COMMAND ----------

print("\n" + "=" * 70)
print("✅ DATA QUALITY VALIDATION COMPLETE!")
print("=" * 70)